# Data Preprocessing

# 1.import libraries and load data

In [1]:
import pandas as pd
df = pd.read_csv(r"C:\Users\sp719\OneDrive\Desktop\project\ipl.csv")

C:\Users\sp719\AppData\Local\Temp\ipykernel_12332\2059052132.py:2: DtypeWarning: Columns (0: season, 1: result_margin) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"C:\Users\sp719\OneDrive\Desktop\project\ipl.csv")


# 2.handle missing values

In [2]:
print(df.isnull().sum())

match_id                  0
season                    0
date                      0
city                  12397
venue                     0
team1                     0
team2                     0
toss_winner               0
toss_decision             0
winner                 4702
result                    0
result_margin             0
player_of_match         806
inning                    0
batting_team              0
bowling_team              0
over                      0
ball                      0
batter                    0
bowler                    0
non_striker               0
runs_scored               0
extras                    0
current_score             0
wickets_down              0
balls_remaining           0
wickets_remaining         0
current_run_rate          0
required_run_rate    144302
target_score         144302
wicket_kind          264382
player_out           264382
fielder              268192
dtype: int64


In [3]:
#for a simple preprocessing approach
df['city'] = df['city'].fillna('Unknown')
df['winner'] = df['winner'].fillna('Unknown')
df['player_of_match'] = df['player_of_match'].fillna('Unknown')#missing text values are replaced with unknown or none
df['wicket_kind'] = df['wicket_kind'].fillna('None')
df['player_out'] = df['player_out'].fillna('None')
df['fielder'] = df['fielder'].fillna('None')

df['required_run_rate'] = df['required_run_rate'].fillna(0)
df['target_score'] = df['target_score'].fillna(0)#numerical values are replaced with 0

# 3.label encoding

In [4]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['toss_decision'] = le.fit_transform(df['toss_decision'])

print(df['toss_decision'].head())

0    1
1    1
2    1
3    1
4    1
Name: toss_decision, dtype: int64


# 4.one hot encoding

In [5]:
df = pd.get_dummies(
    df,
    columns=['team1', 'team2', 'toss_winner', 'venue'],
    dtype=int
)

print(df.head())

   match_id   season       date       city  toss_decision  \
0    335982  2007/08  4/18/2008  Bangalore              1   
1    335982  2007/08  4/18/2008  Bangalore              1   
2    335982  2007/08  4/18/2008  Bangalore              1   
3    335982  2007/08  4/18/2008  Bangalore              1   
4    335982  2007/08  4/18/2008  Bangalore              1   

                  winner result result_margin player_of_match  inning  ...  \
0  Kolkata Knight Riders   runs           140     BB McCullum       1  ...   
1  Kolkata Knight Riders   runs           140     BB McCullum       1  ...   
2  Kolkata Knight Riders   runs           140     BB McCullum       1  ...   
3  Kolkata Knight Riders   runs           140     BB McCullum       1  ...   
4  Kolkata Knight Riders   runs           140     BB McCullum       1  ...   

  venue_Shaheed Veer Narayan Singh International Stadium  \
0                                                  0       
1                                           

# 5.feature scaling

In [10]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# Features for scaling
features = [
    'runs_scored',
    'extras',
    'current_score',
    'wickets_down',
    'balls_remaining',
    'wickets_remaining',
    'current_run_rate',
    'required_run_rate',
    'target_score'
]

# Replace infinity values with NaN
df[features] = df[features].replace([np.inf, -np.inf], np.nan)

# Fill missing values with median
df[features] = df[features].fillna(df[features].median())

# Apply Standard Scaling
scaler = StandardScaler()
df[features] = scaler.fit_transform(df[features])

# Check result
print(df[features].head())

   runs_scored    extras  current_score  wickets_down  balls_remaining  \
0    -0.773651  2.717030      -1.523496     -1.170862         1.688709   
1    -0.773651 -0.198149      -1.523496     -1.170862         1.659399   
2    -0.773651  2.717030      -1.503479     -1.170862         1.659399   
3    -0.773651 -0.198149      -1.503479     -1.170862         1.630088   
4    -0.773651 -0.198149      -1.503479     -1.170862         1.600778   

   wickets_remaining  current_run_rate  required_run_rate  target_score  
0           1.170862         -0.682862           -0.48207     -0.931502  
1           1.170862         -1.916371           -0.48207     -0.931502  
2           1.170862         -0.682862           -0.48207     -0.931502  
3           1.170862         -1.505201           -0.48207     -0.931502  
4           1.170862         -1.916371           -0.48207     -0.931502  


# 6.feature selection
feature selection means choosing the most useful columns for predicting the target

In [12]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif

# Features
features = [
    'runs_scored',
    'extras',
    'current_score',
    'wickets_down',
    'balls_remaining',
    'wickets_remaining',
    'current_run_rate',
    'required_run_rate',
    'target_score'
]

# Input and target
X = df[features]
y = df['winner']

# Feature selection - select top 5
selector = SelectKBest(score_func=f_classif, k=5)

# Fit selector
X_selected = selector.fit_transform(X, y)

# Selected feature names
selected_features = X.columns[selector.get_support()]

print("Selected Features:")
print(selected_features)

# Feature scores
feature_scores = pd.DataFrame({
    'Feature': features,
    'Score': selector.scores_
})

print("\nFeature Scores:")
print(feature_scores.sort_values('Score', ascending=False))

Selected Features:
Index(['current_score', 'wickets_down', 'wickets_remaining',
       'current_run_rate', 'target_score'],
      dtype='str')

Feature Scores:
             Feature       Score
6   current_run_rate  338.550911
2      current_score   89.376033
8       target_score   39.999595
5  wickets_remaining   27.422526
3       wickets_down   27.422526
7  required_run_rate   27.021335
0        runs_scored   14.007324
4    balls_remaining    5.136139
1             extras    2.604149
